In [ ]:
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.models as models
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modèle du GPU : {torch.cuda.get_device_name(0)}")

In [ ]:
def to_dataframe(path):
    df = pd.read_csv(path)
    label_map = {
        "SNE": 0,
        "LY": 1,
        "MO": 2,
        "EO": 3,
        "BA": 4,
        "VLY": 5,
        "MMY": 6,
        "MY": 7,
        "PMY": 8,
        "BL": 9,
        "PC": 10,
        "PLY": 11,
        "BNE": 12,
    }
    if "label" in df.columns:
        df["label_idx"] = df["label"].map(label_map)
    else:
        df["label"] = "undefined"
        df["label_idx"] = -1
    return df


df_train = to_dataframe("/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train_metadata.csv")
df_test = to_dataframe("/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test_metadata.csv")

df_train['fold'] = -1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['label'])):
    df_train.loc[val_idx, 'fold'] = fold

In [ ]:
## This section is for data manipulation ##

class WhiteBloodCellDataset(Dataset):
    def __init__(self, dataframe, path, transform=None):
        self.names = dataframe["ID"].values
        self.labels = dataframe["label_idx"].values
        self.path = path
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        image_path = self.path + str(self.names[idx])
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label


def load_data(df_train, df_test, current_fold, train_data_path, test_data_path):
    # Resize to 384x384 for ResNet + Data Augmentation
    train_transforms = transforms.Compose(
        [
            transforms.Resize((384, 384)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=180),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    val_test_transforms = transforms.Compose(
        [
            transforms.Resize((384, 384)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    # --- SÉPARATION TRAIN / VAL DU FOLD ---
    train_df = df_train[df_train['fold'] != current_fold].copy().reset_index(drop=True)
    val_df = df_train[df_train['fold'] == current_fold].copy().reset_index(drop=True)

    # --- 1. GÉNÉRALISTE ---
    label_map = {
        "SNE": 0, "LY": 1, "MO": 2, "EO": 3, "BA": 4, "VLY": 5,
        "MMY": 6, "MY": 7, "PMY": 8, "BL": 9, "PC": 10, "PLY": 11,
        "BNE": 12
    }

    train_df = train_df.copy()
    val_df = val_df.copy()

    train_df['label_idx'] = train_df['label'].map(label_map)
    val_df['label_idx'] = val_df['label'].map(label_map)

    # Datasets
    train_set = WhiteBloodCellDataset(dataframe=train_df, path=train_data_path, transform=train_transforms)
    val_set = WhiteBloodCellDataset(dataframe=val_df, path=train_data_path, transform=val_test_transforms)
    test_set = WhiteBloodCellDataset(dataframe=df_test, path=test_data_path, transform=val_test_transforms)
    
    # Sampler
    counts = train_df["label_idx"].value_counts()
    weights = 1.0 / counts
    samples_weights = torch.from_numpy(train_df["label_idx"].map(weights).values).double()
    sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

    # DataLoaders
    train_loader = DataLoader(train_set, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader

In [ ]:
## This section is for model management ##

def get_model():
    # ResNet model
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 13)

    return model


def train_model(
    model, device, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, patience, path
):
    best_f1 = 0
    counter = 0
    history = {
        "train_loss": [],
        "val_f1": [],
    }

    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Stats
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        history["train_loss"].append(epoch_loss)
        print(
            f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%"
        )

        # Early Stopping
        current_f1 = validate_model(model, device, val_loader)
        scheduler.step(current_f1)
        history["val_f1"].append(current_f1)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Validation Macro-F1: {current_f1:.4f}")

        if current_f1 > best_f1:
            best_f1 = current_f1
            torch.save(model.state_dict(), path)
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f"Early Stopping !")
            break

    print("Training finished !")
    return history


def validate_model(model, device, test_loader):
    model.eval()
    all_pred = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_pred.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    score = f1_score(all_labels, all_pred, average="macro")
    return score


def predict_model(model, device, test_loader, inv_label_map):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())

    return [inv_label_map[p] for p in all_preds]

In [ ]:
## This section is for training/validation/testing ##

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using : {device}")

# Loading data
train_dir = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train/"
test_dir = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test/"

scores = []

for fold in range(5):
    print(f"Starting fold {fold} !")
    train_loader, val_loader, test_loader = load_data(
        df_train, df_test, fold, train_dir, test_dir
    )
    print("Data loaded !")
    save_path = f"best_white_cell_model.pth_{fold}"
    # Model
    model = get_model()
    model = model.to(device)
    
    # Parameters
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)
    num_epochs = 40
    patience = 5
    
    # Train
    history = train_model(
        model, device, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, patience, save_path
    )
    model.load_state_dict(torch.load(save_path))
    print("Model saved !")
    
    # Validation
    score = validate_model(model, device, val_loader)
    print(f"The Macro F1-score for Validation is {score}.")
    scores.append(score)

print(f"\n K-FOLD Done ! Mean score : {sum(scores)/len(scores):.4f}")

In [ ]:
inv_label_map_final = {
    0: "SNE", 1: "LY", 2: "MO", 3: "EO", 4: "BA", 5: "VLY", 
    6: "MMY", 7: "MY", 8: "PMY", 9: "BL", 10: "PC", 11: "PLY", 
    12: "BNE" 
}

resnet_models = []
for i in range(5):
    save_path = f"best_white_cell_model.pth_{i}"
    model = get_model()
    model.load_state_dict(torch.load(save_path))
    model = model.to(device)
    model.eval()
    resnet_models.append(model)

print("Models loaded ! !")

all_preds_strings = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)

        batch_probs_accumulated = torch.zeros(images.size(0), 13).to(device)
        
        for model in resnet_models:
            out1 = model(images)
            out2 = model(torch.flip(images, dims=[3])) 
            out3 = model(torch.flip(images, dims=[2])) 
            out4 = model(torch.rot90(images, k=2, dims=[2, 3])) 
            
            outputs_tta_avg = (out1 + out2 + out3 + out4) / 4.0
            
            probs = F.softmax(outputs_tta_avg, dim=1)
            
            batch_probs_accumulated += probs
            
        batch_probs_final = batch_probs_accumulated / len(resnet_models)
        
        _, predicted_classes = torch.max(batch_probs_final, 1)
        
        for idx in predicted_classes:
            all_preds_strings.append(inv_label_map_final[idx.item()])

    sample_sub = pd.read_csv("/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/sample_submission.csv")
    if len(sample_sub) == len(all_preds_strings):
        sample_sub = sample_sub.drop(columns=["Unnamed: 0"])
        sample_sub["label"] = all_preds_strings
        sample_sub.to_csv("submission.csv", index=False)
        print("Fichier submission.csv prêt !")
    else:
        print(f"Size error ! Got: {len(all_preds_strings)} instead of: {len(sample_sub)}")
    print("Model tested !")

In [ ]:
import matplotlib.pyplot as plt

def plot_training_curve(train_losses, val_f1s):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Train Loss', color='tab:red')
    ax1.plot(train_losses, color='tab:red', label='Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Val Macro-F1', color='tab:blue')
    ax2.plot(val_f1s, color='tab:blue', label='Macro-F1')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    plt.title('Performance during training')
    plt.show()

In [ ]:
plot_training_curve(history["train_loss"], history["val_f1"])

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, device, val_loader, labels_names):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels_names, yticklabels=labels_names)
    plt.xlabel('Predictions')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix (Validation Set)')
    plt.show()

In [ ]:
plot_confusion_matrix(model, device, val_loader, ["SNE", "LY", "MO", "EO", "BA", "VLY", "MMY", "MY", "PMY", "BL", "PC", "PLY", "BNE"] )